# Annotation pipeline — run register

A living record of the rewritten plasmid annotation pipeline. **Every number here is read from a
pipeline output file — nothing is typed by hand.** That is deliberate: the round-1 review found
that the previous Arc-2 numbers existed only as hand-written prose in notebooks saved without
outputs, and so could never be checked against the code that produced them.

| | |
|---|---|
| Pipeline | `workflow/Snakefile` (Snakemake, lint-clean) |
| Design | `plans/2026-09-02-annotation-pipeline-design.md` |
| Logic under test | `src/plasmidann/` — 21 tests via `pytest` |
| Repo | `git@github.com:mujkicamar-bio/Kryptos.git` (private) |

In [1]:
import csv, subprocess, collections, pathlib
ROOT = pathlib.Path("/gorilla/proj/h-mel-phylo/h-mel-phylo/private/amar/plasmid_analyis")
R = ROOT / "results"

def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True, cwd=ROOT).stdout.strip()

print("git commit :", sh("git rev-parse --short HEAD"), "|", sh("git log -1 --format=%s"))
print()
print("config/config.yaml"); print(sh("cat config/config.yaml"))
print(); print(sh("grep min_explained config/cascade.yaml"))

git commit : 77a62d0 | Call ORFs down to 20 aa (was pyrodigal's 30 aa default)

config/config.yaml
working_set_fasta: data/plasmidscope_primary/working_set.fna.gz
master_table: data/plasmidscope_primary/plasmid_metadata_master.tsv
outdir: results
n_shards: 600
exclude_hab_top: [Simulated-artifact, Lab-artifact]
min_orf_aa: 20

min_explained: 0.5


## 1 · Run register — every SLURM job, and what happened to it

In [2]:
print(sh("sacct -u $USER -S 2026-09-02 --format=JobID,JobName%14,State,Elapsed,ExitCode -X").strip())
print()
print("queued now:")
print(sh('squeue -u $USER -o "%.10i %.13j %.9T %S %R"') or "  (nothing queued)")

JobID               JobName      State    Elapsed ExitCode 
------------ -------------- ---------- ---------- -------- 
6595863          recl_union  COMPLETED   00:15:49      0:0 
6609285        plasmid_s1s2     FAILED   00:10:11      1:0 
6609649        plasmid_s1s2 CANCELLED+   00:00:23      0:0 
6613795       plasmid_annot CANCELLED+   00:00:00      0:0 
6613828       plasmid_annot    PENDING   00:00:00      0:0

queued now:
JOBID          NAME     STATE START_TIME NODELIST(REASON)
   6613828 plasmid_annot   PENDING 2026-09-07T04:30:00 (Priority)


### What each run did

- **6609285** — first full attempt. S1+S2 completed in ~9 min; failed at tier T1 with exit 127
  (`hmmsearch` not on PATH — the batch job did not load the HMMER module). The S1/S2 figures in
  section 2 come from this run.
- **6609649** — brief `haswell` submission, cancelled on request; produced nothing.
- **6613795** — cancelled before starting; it would have reused the 30 aa ORF set.
- **6613828** — the full run: 600 ORF calls at the new 20 aa floor plus the 8-tier cascade.

## 2 · S1 / S2 — ORF calling and dereplication

In [3]:
idx = R/"s1/orf_index.tsv"
if not idx.exists():
    print("S1 not complete for the current run")
else:
    n = 0; per = collections.Counter(); lens = []
    with open(idx, newline="") as fh:
        for r in csv.DictReader(fh, delimiter="\t"):
            n += 1; per[r["plasmid_id"]] += 1; lens.append(len(r["seq"]))
    lens.sort()
    uniq = sum(1 for _ in open(R/"s2/protein_map.tsv"))
    print(f"plasmids in analysis set : {len(per):>10,}")
    print(f"ORFs called              : {n:>10,}")
    print(f"unique protein sequences : {uniq:>10,}")
    print(f"exact-duplicate collapse : {100*(1-uniq/n):>9.1f}%")
    print()
    print(f"ORFs per plasmid   mean {n/len(per):.1f}   median {sorted(per.values())[len(per)//2]}")
    print(f"protein length aa  median {lens[len(lens)//2]}   q25 {lens[len(lens)//4]}   q75 {lens[3*len(lens)//4]}")
    print(f"under 30 aa        {100*sum(1 for l in lens if l<30)/len(lens):.2f}%")
    print(f"shortest           {lens[0]} aa")

plasmids in analysis set :    143,479
ORFs called              :  9,285,486
unique protein sequences :  3,487,147
exact-duplicate collapse :      62.4%

ORFs per plasmid   mean 64.7   median 13
protein length aa  median 203   q25 111   q75 331


under 30 aa        0.14%
shortest           20 aa


**The 62.4% collapse is the load-bearing number so far.** The design assumed 40–50%; a
200-plasmid smoke test suggested 9%. At full scale nearly two-thirds of the plasmid proteome is
exact duplicates, so the cascade searches ~3.5M sequences rather than ~9.3M — a 2.7× cost
reduction before any search runs. It also sharpens the redundancy problem the round-1 devil's
advocate raised: this corpus is far more redundant at protein level than at plasmid level, which
matters for any later claim about family prevalence.

## 3 · S3 — the annotation cascade, tier by tier

In [4]:
rows = []
for t in [f"T{i}" for i in range(1, 9)]:
    h, u = R/f"s3/{t}/hits.tsv", R/f"s3/{t}/unresolved.faa"
    if not h.exists():
        continue
    nh = sum(1 for _ in open(h)) - 1
    nu = sum(1 for l in open(u) if l[0] == ">") if u.exists() else 0
    rows.append((t, nh, nu))

if not rows:
    print("cascade has not run yet")
else:
    print(f"{'tier':<6}{'resolved':>12}{'still open':>13}")
    for t, nh, nu in rows:
        print(f"{t:<6}{nh:>12,}{nu:>13,}")

cascade has not run yet


A tier only sees what earlier tiers could not explain, and a protein is handed on until
**`explained_fraction >= 0.5`** — so a small domain hit no longer terminates the search over the
rest of the protein.

## 4 · S4 — the deliverable: annotated plasmids

In [5]:
f = R/"s4/plasmid_annotation.tsv"
if not f.exists():
    print("S4 not yet produced")
else:
    fc = collections.Counter(); comp = collections.Counter(); tier = collections.Counter(); n = 0
    with open(f, newline="") as fh:
        for r in csv.DictReader(fh, delimiter="\t"):
            n += 1
            fc[r["functional_class"]] += 1
            comp[r["annot_completeness"]] += 1
            tier[r["annot_tier"] or "unresolved"] += 1
    print(f"ORFs annotated: {n:,}\n")
    for title, c in (("functional_class", fc), ("annot_completeness", comp),
                     ("resolved at tier", tier)):
        print(title)
        for k, v in c.most_common():
            print(f"   {str(k):<26}{v:>11,}  {100*v/n:5.1f}%")
        print()

S4 not yet produced


`UNCHARACTERIZED_HOMOLOG` is the screening target: homologs exist, sometimes many and across
taxa, but every one of them is itself unnamed. Certainly real, certainly unknown.
`annot_completeness` keeps a 0.51 explanation from being counted as a 0.99 one.

## 5 · Reproduce

```bash
cd /gorilla/proj/h-mel-phylo/h-mel-phylo/private/amar/plasmid_analyis
.venv/bin/python -m pytest -q                    # 21 tests
.venv/bin/snakemake -n -s workflow/Snakefile     # dry run
sbatch workflow/run_s1s2.sbatch                  # full run
```

Snakemake resumes: a job that hits its walltime can simply be resubmitted and it continues from
the last completed tier. Changing any pre-registered threshold in `config/` invalidates every
downstream output automatically — that is how the 20 aa change forced all 600 ORF-calling jobs to
rerun rather than leaving stale results in place.